In [0]:
from pyspark.sql import functions as F

In [0]:
SOURCE_CATALOG_NAME = 'covid_19'
SOURCE_SCHEMA_NAME = 'bronze'
SOURCE_TABLE_NAME = 'vaccinations'

TARGET_CATALOG_NAME = 'covid_19'
TARGET_SCHEMA_NAME = 'silver'
TARGET_TABLE_NAME = 'vaccinations'

In [0]:
INVALID_COUNTRY_VALUES = [
    'Africa', 'Asia', 'Europe', 'European Union', 'High income', 
    'Low income', 'Lower middle income', 'North America', 'Oceania', 
    'South America', 'Upper middle income', 'World'
]

df = (
    spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.{SOURCE_TABLE_NAME}')
    .select(
        F.col('country'),
        F.col('iso_code'),
        F.explode('data').alias('vaccination_data')
    )
    .select(
        F.col('country'),
        F.col('iso_code'),
        F.to_date(F.col('vaccination_data.date')).alias('date'),

        F.col('vaccination_data.total_vaccinations').cast('long').alias('total_vaccinations'),
        F.col('vaccination_data.people_vaccinated').cast('long').alias('people_vaccinated'),
        F.col('vaccination_data.people_fully_vaccinated').cast('long').alias('people_fully_vaccinated'),
        F.col('vaccination_data.total_boosters').cast('long').alias('total_boosters'),
        F.col('vaccination_data.daily_vaccinations').cast('long').alias('daily_vaccinations'),
        F.col('vaccination_data.daily_people_vaccinated').cast('long').alias('daily_people_vaccinated')
    )
    .filter(F.col('country').isNotNull())
    .filter(F.col('iso_code').isNotNull())
    .filter(F.col('date').isNotNull())
    .filter(~F.col('country').isin(INVALID_COUNTRY_VALUES))
)

In [0]:
df.write\
    .mode('overwrite')\
    .saveAsTable(f'{TARGET_CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TARGET_TABLE_NAME}')